# 00 - Vérification de l'installation

À exécuter **une fois par membre**, puis à chaque fois que quelque chose semble cassé.
Toutes les cellules doivent afficher `OK`. Sinon, le message indique quoi corriger.

**Avant de lancer :** en haut à droite, *Select Kernel > Colab* et choisissez un serveur
(CPU suffit ici ; GPU T4 pour vérifier aussi le GPU). Un seul serveur à la fois par compte.

## 1. Configuration commune (identique aux autres notebooks)

Sur Colab, une fenêtre demande l'accès à Google Drive : acceptez.

In [8]:
# --- Configuration commune (identique dans chaque notebook) ---
from pathlib import Path
import random

import numpy as np

# Exécution sur Google Colab ou en local : le même notebook fonctionne dans les deux cas.
try:
    import google.colab  # noqa: F401  (ce module n'existe que sur Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Sur Colab, tout ce qui est écrit dans /content disparaît à la fin de la session.
    # On travaille donc dans Google Drive : données, figures et poids y sont conservés.
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/AutoMaze_Ai")  # TODO(équipe) : adapter au dossier partagé de l'équipe
else:
    # En local, ce notebook est dans notebooks/ : la racine du projet est le dossier parent.
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Colab" if IN_COLAB else "Local", "| racine du projet :", ROOT)
DATA_DIR = ROOT / "data"          # données locales, ignorées par Git
RAW_DIR = DATA_DIR / "raw"        # fichiers téléchargés tels quels (ex. annotations COCO)
CROPS_DIR = DATA_DIR / "crops"    # recadrages produits par le notebook 02
MODELS_DIR = ROOT / "models"      # poids + métadonnées (les .pt sont ignorés par Git)
FIGURES_DIR = ROOT / "reports" / "figures"  # figures à reprendre dans le rapport

for d in (RAW_DIR, CROPS_DIR, MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ordre des classes = source de vérité pour TOUT le projet (notebooks + simulateur).
# Ne le changez plus une fois l'entraînement commencé : l'indice 0 doit toujours désigner la même classe.
CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]
CLASSES_FR = ["Personne", "Vélo", "Voiture", "Moto", "Bus", "Camion"]

# Graine aléatoire : rend les résultats reproductibles (même découpage, mêmes initialisations).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab | racine du projet : /content/drive/MyDrive/AutoMaze_Ai


## 2. Le dossier du projet est-il trouvé dans Drive ?

In [9]:
if IN_COLAB:
    assert Path("/content/drive/MyDrive").exists(), (
        "Drive n'est pas monté. Palette de commandes (Ctrl+Shift+P) > « Colab: Mount Google Drive to Server », "
        "exécutez la cellule ajoutée, puis relancez la cellule 1."
    )
    assert ROOT.exists(), (
        f"{ROOT} introuvable. Si le dossier vous a été partagé : clic droit sur AutoMaze_Ai dans Drive > "
        "Organiser > Ajouter un raccourci > Mon Drive."
    )
print("OK — dossier du projet :", ROOT)
print("Contenu :", sorted(p.name for p in ROOT.iterdir()))

OK — dossier du projet : /content/drive/MyDrive/AutoMaze_Ai
Contenu : ['data', 'models', 'reports']


## 3. Peut-on écrire dans le dossier partagé ?

In [10]:
import getpass
from datetime import datetime

check_file = ROOT / "reports" / f"verification_{getpass.getuser()}.txt"
check_file.write_text(f"Vérification réussie le {datetime.now():%Y-%m-%d %H:%M}\n", encoding="utf-8")
assert check_file.read_text(encoding="utf-8").startswith("Vérification"), "Écriture impossible dans le dossier."
print("OK — fichier écrit puis relu :", check_file)

OK — fichier écrit puis relu : /content/drive/MyDrive/AutoMaze_Ai/reports/verification_root.txt


## 4. Bibliothèques et matériel

In [11]:
import platform
import shutil

import cv2
import sklearn
import torch
import torchvision

print("Python      ", platform.python_version())
print("PyTorch     ", torch.__version__, "| torchvision", torchvision.__version__)
print("scikit-learn", sklearn.__version__, "| OpenCV", cv2.__version__)

if torch.cuda.is_available():
    print("OK — GPU disponible :", torch.cuda.get_device_name(0))
else:
    print("Pas de GPU sur ce serveur : normal pour les notebooks 01 à 03.",
          "Pour le 04, prenez un serveur GPU (T4).")

disk = shutil.disk_usage("/content" if IN_COLAB else ROOT)
print(f"Espace disque libre : {disk.free / 1e9:.1f} Go")

Python       3.13.16
PyTorch      2.11.0+cpu | torchvision 0.26.0+cpu
scikit-learn 1.6.1 | OpenCV 5.0.0
Pas de GPU sur ce serveur : normal pour les notebooks 01 à 03. Pour le 04, prenez un serveur GPU (T4).
Espace disque libre : 93.6 Go


## 5. Annotations COCO déjà téléchargées ?

Un seul membre les télécharge (notebook 01) : elles apparaissent ensuite pour toute l'équipe.

In [12]:
ann = RAW_DIR / "coco" / "annotations" / "instances_train2017.json"
if ann.exists():
    print(f"OK — annotations présentes ({ann.stat().st_size / 1e6:.0f} Mo)")
else:
    print("Pas encore d'annotations : à faire dans le notebook 01, section 1.")

Pas encore d'annotations : à faire dans le notebook 01, section 1.


## Tout est OK ?

Notez-le dans le journal de bord du rapport, puis fermez le serveur si vous avez fini
(*Ctrl+Shift+P > Colab: Remove Server*) pour ne pas consommer votre quota.